# SQL Examples

Connect to system

In [2]:
%connect vsdemo

Connected: 'vsdemo' connection activated for user 'df120645'


### Base table with data

In [3]:
SHOW TABLE commercial_real_estate_III

,Request Text
1,"CREATE MULTISET TABLE DF120645.commercial_real_estate_III ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, ""Unnamed: 0"" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, ""title"" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, nbn VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, address VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, text VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, area VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, ""type"" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, lattitude FLOAT, longitude FLOAT) NO PRIMARY INDEX ;"


Drop if already created

In [9]:
DROP TABLE vectorstore_commercial_real_estate_II_index

Success: 20 rows affected

### Create embedding index table
NOTE:
- Multiple text and value columns are concatenated together to provide text for embedding
- Convention of vectorstore_\<name\>_index is used to match tables created using teradatagenai python
- AWSEmbeddingsAuth was established using the teradatagenai python package
- The Accumulate argument is used to copy fields into the actual index table for efficient filtering

In [10]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_II_index AS (
    SELECT id, price, Embedding as Vector_Index, Message 
    FROM AI_TEXTEMBEDDINGS(   
            ON (
                SELECT 
                    dt.id, 
                    CONCAT(
                        dt.text, ' ', 
                        dt.address, ' ', 
                        'Latitude: ', CAST(dt.lattitude AS VARCHAR(50)), ' ', 
                        'Longitude: ', CAST(dt.longitude AS VARCHAR(50))
                    ) AS text, 
                    dt.price, 
                    dt."title", 
                    td_byone() 
                FROM commercial_real_estate_iii dt 
                SAMPLE 1
            ) AS InputTable PARTITION BY TD_BYONE()
            USING authorization(AWSEmbeddingsAuth)
            TextColumn('text')
            ApiType('aws')
            REGION('us-west-2')
            ModelName('amazon.titan-embed-text-v1')
            outputformat('vector')
            Accumulate(' "id" ', ' "title" ', ' "price" ')
    ) AS dt
) WITH DATA PRIMARY INDEX (id);

Success: 0 rows affected

In [14]:
SHOW TABLE vectorstore_commercial_real_estate_II_index

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_II_index ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, Vector_Index SYSUDTLIB.Vector, Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC) PRIMARY INDEX ( id );"


In [ ]:
DROP TABLE vectorstore_commercial_real_estate_II_model

# Creating the models
### The models for HNSW / Nearest Neighbor and KMeans / IVF are shown. 
### Vector Distance does not have a model, rather it is a full table scan of the index and should be used on small tables

## Create HNSW Model

In [15]:
SELECT * FROM TD_HNSW (
  ON vectorstore_commercial_real_estate_II_index AS InputTable  
  OUT PERMANENT TABLE ModelTable(vectorstore_commercial_real_estate_II_model)
  USING
  IdColumn('id')
  VectorColumn('vector_index')   
  Seed(0)
  EfConstruction(32)
  NumConnPerNode(32)
  MaxNumConnPerNode(32)  
  DistanceMeasure('EUCLIDEAN')
  EmbeddingSize(1536)  
  ApplyHeuristics('True')
) as dt;

Success: 0 rows affected

,message
1,HNSW Graph Construction completed successfully and stored in the OutputTable.


## Create KMeans/LVF Model

In [4]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_kmeans_model as (
SELECT * FROM TD_KMEANS (
  ON vectorstore_commercial_real_estate_II_index AS InputTable
  USING
  IdColumn('id')
  TargetColumns('Vector_Index')
  InitialCentroidsMethod('RANDOM')
  NumClusters(10)
  Seed(0)
  StopThreshold(0.0395)
  MaxIterNum(10)
  NumInit(1)
  EmbeddingSize(1536)
) AS dt) WITH DATA NO PRIMARY INDEX

Connection failure detected. Reconnect required for connection: vsdemo.
Re-enter Password ········


In [5]:
SHOW TABLE vectorstore_commercial_real_estate_kmeans_model

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_kmeans_model ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( td_clusterid_kmeans BIGINT, vector_index SYSUDTLIB.Vector, td_size_kmeans BIGINT, td_withinss_kmeans FLOAT, id BYTEINT, td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC) NO PRIMARY INDEX ;"


## Create Centroids

In [8]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids AS (
SELECT id, Vector_Index, td_clusterid_kmeans as clusterID FROM TD_KMEANSPREDICT(
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  ON vectorstore_commercial_real_estate_kmeans_model AS ModelTable DIMENSION
  USING
  Accumulate( 'Vector_Index')
) AS dt) WITH DATA PRIMARY INDEX (clusterID);

Success: 0 rows affected

Drop if existing

In [7]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids

Success: 20 rows affected

# Vector Similarity Search

### Vector Distance Similarity Search

### HNSW Similarity Search

### KMeans Similarity Search